# GameGuide — Data Cleaning

## Step 2: Data Cleaning

The objective of this step is to transform the raw Steam Games
dataset into a clean, typed, structured dataset suitable for
exploratory analysis, feature engineering, search, recommendation,
and chatbot development.

The cleaning process includes:

1. Removing unusable rows
2. Parsing list-like columns
3. Converting dates and numerical fields
4. Parsing estimated owners
5. Cleaning text fields
6. Standardizing rating fields
7. Saving the cleaned dataset as Parquet

In [4]:
import ast
import html
import re
import numpy as np
import pandas as pd

In [5]:
DATA_PATH = "../data/raw/games_march2025_full.csv"
df = pd.read_csv(DATA_PATH, low_memory=False)
print("Raw Data Loaded Successfully!")
print("Shape: ",df.shape)

Raw Data Loaded Successfully!
Shape:  (94948, 47)


In [6]:
cleaning_log = []
def log_step(step, before, after):
    cleaning_log.append({
        "step": step,
        "rows_before": before,
        "rows_after": after,
        "rows_removed": before - after
    })

In [8]:
cleaning_log_df = pd.DataFrame(cleaning_log)

In [9]:
before = len(df)
df = df.dropna(subset=["name", "appid"]).copy()
after = len(df)
log_step(
    "Remove rows with missing name/appid",
    before,
    after
)
print("Rows before:", before)
print("Rows after:", after)
print("Rows removed:", before - after)

Rows before: 94948
Rows after: 94946
Rows removed: 2


In [10]:
before = len(df)
df = df.drop_duplicates(subset=["appid"], keep="first").copy()
after = len(df)
log_step(
    "Remove duplicate appid values",
    before,
    after
)
print("Rows before:", before)
print("Rows after:", after)
print("Rows removed:", before - after)

Rows before: 94946
Rows after: 94946
Rows removed: 0


In [11]:
non_game_pattern = (
    r"\b(?:Soundtrack|OST|Demo|Playtest|Dedicated Server|"
    r"Artbook|Wallpaper|Beta)\b"
)

In [12]:
non_game_mask = df["name"].str.contains(
    non_game_pattern,
    case=False,
    na=False,
    regex=True
)

non_game_entries = df.loc[
    non_game_mask,
    ["appid", "name", "price"]
]

print("Potential non-game entries:", len(non_game_entries))
non_game_entries.head(30)

Potential non-game entries: 5396


,appid,name,price
19,1952710,BACKGAMMON Playtest,0.0
22,2912110,Baldur's Gate 3 Playtest,0.0
57,1584640,NARAKA: BLADEPOINT Playtest,0.0
69,2936120,Marvel Rivals Playtest,0.0
93,1489280,鬼谷八荒 Playtest,0.0
122,3341250,World of Tanks Blitz Playtest,0.0
148,1611740,BattleBit Remastered Playtest,0.0
158,1469850,Inscryption Playtest,0.0
166,1990390,Splitgate Playtest,0.0
171,2460470,Balatro Playtest,0.0


In [13]:
non_game_entries.head(30)

,appid,name,price
19,1952710,BACKGAMMON Playtest,0.0
22,2912110,Baldur's Gate 3 Playtest,0.0
57,1584640,NARAKA: BLADEPOINT Playtest,0.0
69,2936120,Marvel Rivals Playtest,0.0
93,1489280,鬼谷八荒 Playtest,0.0
122,3341250,World of Tanks Blitz Playtest,0.0
148,1611740,BattleBit Remastered Playtest,0.0
158,1469850,Inscryption Playtest,0.0
166,1990390,Splitgate Playtest,0.0
171,2460470,Balatro Playtest,0.0


In [14]:
patterns = {
    "Soundtrack": r"\bSoundtrack\b",
    "OST": r"\bOST\b",
    "Demo": r"\bDemo\b",
    "Playtest": r"\bPlaytest\b",
    "Dedicated Server": r"\bDedicated Server\b",
    "Artbook": r"\bArtbook\b",
    "Wallpaper": r"\bWallpaper\b",
    "Beta": r"\bBeta\b"
}

for label, pattern in patterns.items():
    count = df["name"].str.contains(
        pattern,
        case=False,
        na=False,
        regex=True
    ).sum()

    print(f"{label}: {count}")

Soundtrack: 2
OST: 1
Demo: 61
Playtest: 5217
Dedicated Server: 11
Artbook: 1
Wallpaper: 21
Beta: 102


In [15]:
before = len(df)
df = df.loc[~non_game_mask].copy()
after = len(df)
log_step(
    "Remove non-game entries",
    before,
    after
)
print("Rows before:", before)
print("Rows after:", after)
print("Rows removed:", before - after)

Rows before: 94946
Rows after: 89550
Rows removed: 5396


In [16]:
no_description = (
    df["short_description"].isna()
    & df["about_the_game"].isna()
    & df["detailed_description"].isna()
)

no_genres = df["genres"].isna()

no_useful_content = no_description & no_genres

print("Rows with no description:", no_description.sum())
print("Rows with no genres:", no_genres.sum())
print("Rows with neither description nor genres:", no_useful_content.sum())

Rows with no description: 83
Rows with no genres: 0
Rows with neither description nor genres: 0


In [17]:
before = len(df)
# No rows satisfy the condition:
# missing all description fields AND missing genres.
df = df.loc[~no_useful_content].copy()
after = len(df)
log_step(
    "Remove entries with no description and no genres",
    before,
    after
)
print("Rows before:", before)
print("Rows after:", after)
print("Rows removed:", before - after)

Rows before: 89550
Rows after: 89550
Rows removed: 0


In [18]:
def parse_list(x):
    if pd.isna(x):
        return []

    try:
        v = ast.literal_eval(x)

        if isinstance(v, dict):
            return list(v.keys())

        return list(v) if isinstance(v, (list, tuple, set)) else [str(v)]

    except Exception:
        return [s.strip() for s in str(x).split(",") if s.strip()]

In [19]:
test_values = [
    "['Action', 'RPG']",
    "['Multi-player', 'Co-op']",
    "{'FPS': 100, 'Shooter': 80}",
    np.nan,
    "Action, RPG"
]

for value in test_values:
    print(value, "→", parse_list(value))

['Action', 'RPG'] → ['Action', 'RPG']
['Multi-player', 'Co-op'] → ['Multi-player', 'Co-op']
{'FPS': 100, 'Shooter': 80} → ['FPS', 'Shooter']
nan → []
Action, RPG → ['Action', 'RPG']


In [20]:
list_columns = [
    "genres",
    "categories",
    "developers",
    "publishers",
    "supported_languages"
]

for col in list_columns:
    df[col] = df[col].apply(parse_list)

In [21]:
for col in list_columns:
    print("=" * 50)
    print(col)
    print("dtype:", df[col].dtype)
    print("Example:", df[col].iloc[0])
    print("Python type:", type(df[col].iloc[0]))

genres
dtype: object
Example: ['Action', 'Free To Play']
Python type: <class 'list'>
categories
dtype: object
Example: ['Multi-player', 'Cross-Platform Multiplayer', 'Steam Trading Cards', 'Steam Workshop', 'In-App Purchases', 'Valve Anti-Cheat enabled', 'Stats', 'Remote Play on Phone', 'Remote Play on Tablet', 'Remote Play on TV', 'Steam Timeline']
Python type: <class 'list'>
developers
dtype: object
Example: ['Valve']
Python type: <class 'list'>
publishers
dtype: object
Example: ['Valve']
Python type: <class 'list'>
supported_languages
dtype: object
Example: ['Czech', 'Danish', 'Dutch', 'English', 'Finnish', 'French', 'German', 'Hungarian', 'Italian', 'Japanese', 'Korean', 'Norwegian', 'Polish', 'Portuguese - Portugal', 'Portuguese - Brazil', 'Romanian', 'Russian', 'Simplified Chinese', 'Spanish - Spain', 'Swedish', 'Thai', 'Traditional Chinese', 'Turkish', 'Bulgarian', 'Ukrainian', 'Greek', 'Spanish - Latin America', 'Vietnamese', 'Indonesian']
Python type: <class 'list'>


In [22]:
for col in list_columns:
    string_count = df[col].apply(lambda x: isinstance(x, str)).sum()
    print(f"{col}: {string_count} string values")

genres: 0 string values
categories: 0 string values
developers: 0 string values
publishers: 0 string values
supported_languages: 0 string values


In [23]:
def parse_tags(x, top_n=15):
    if pd.isna(x):
        return []

    try:
        tags_dict = ast.literal_eval(x)

        if isinstance(tags_dict, dict):
            sorted_tags = sorted(
                tags_dict.items(),
                key=lambda item: item[1],
                reverse=True
            )

            return [tag for tag, votes in sorted_tags[:top_n]]

        return []

    except Exception:
        return []

In [24]:
test_tags = {
    "FPS": 100,
    "Shooter": 80,
    "Action": 70,
    "Multiplayer": 60,
    "RPG": 50
}

print(parse_tags(str(test_tags), top_n=3))

['FPS', 'Shooter', 'Action']


In [25]:
df["tags"] = df["tags"].apply(parse_tags)

In [26]:
print("Example tags:", df["tags"].iloc[0])
print("Python type:", type(df["tags"].iloc[0]))

Example tags: ['FPS', 'Shooter', 'Multiplayer', 'Competitive', 'Action', 'Team-Based', 'e-sports', 'Tactical', 'First-Person', 'PvP', 'Online Co-Op', 'Co-op', 'Strategy', 'Military', 'War']
Python type: <class 'list'>


In [27]:
tag_string_count = df["tags"].apply(
    lambda x: isinstance(x, str)
).sum()

print("String values remaining in tags:", tag_string_count)

String values remaining in tags: 0


In [28]:
print(df["tags"].apply(len).describe())

count    89550.000000
mean         9.435410
std          6.076085
min          0.000000
25%          4.000000
50%         12.000000
75%         15.000000
max         15.000000
Name: tags, dtype: float64


In [29]:
df["release_date"] = pd.to_datetime(
    df["release_date"],
    errors="coerce"
)

df["release_year"] = df["release_date"].dt.year

In [30]:
print("release_date dtype:", df["release_date"].dtype)
print("Missing release dates:", df["release_date"].isna().sum())
print(df[["release_date", "release_year"]].head())

release_date dtype: datetime64[ns]
Missing release dates: 0
  release_date  release_year
0   2012-08-21          2012
1   2017-12-21          2017
2   2013-07-09          2013
3   2015-04-13          2015
4   2015-12-01          2015


In [31]:
df["price"] = pd.to_numeric(
    df["price"],
    errors="coerce"
).fillna(0)

In [32]:
df["is_free"] = df["price"] == 0

In [79]:
import sys
from pathlib import Path

sys.path.append(str(Path.cwd().parent))

In [80]:
from src.config import USD_TO_INR
df["price_inr"] = df["price"] * USD_TO_INR

In [35]:
print(df[["name", "price", "is_free", "price_inr"]].head(10))

                              name  price  is_free  price_inr
0                 Counter-Strike 2   0.00     True     0.0000
1              PUBG: BATTLEGROUNDS   0.00     True     0.0000
2                           Dota 2   0.00     True     0.0000
3        Grand Theft Auto V Legacy   0.00     True     0.0000
4  Tom Clancy's Rainbow Six® Siege  19.99    False  1921.4388
5  Tom Clancy's Rainbow Six® Siege  19.99    False  1921.4388
6  Tom Clancy's Rainbow Six® Siege  19.99    False  1921.4388
7  Tom Clancy's Rainbow Six® Siege  19.99    False  1921.4388
8  Tom Clancy's Rainbow Six® Siege   3.99    False   383.5188
9                  Team Fortress 2   0.00     True     0.0000


In [36]:
print("Free games:", df["is_free"].sum())
print("Paid games:", (~df["is_free"]).sum())

Free games: 14066
Paid games: 75484


In [37]:
def parse_owners(s):
    nums = re.findall(
        r"\d+",
        str(s).replace(",", "")
    )

    return (
        int(nums[0]),
        int(nums[-1])
    ) if nums else (np.nan, np.nan)

In [38]:
test_owners = [
    "100000000 - 200000000",
    "50,000,000 - 100,000,000",
    "0 - 20000",
    np.nan
]

for value in test_owners:
    print(value, "→", parse_owners(value))

100000000 - 200000000 → (100000000, 200000000)
50,000,000 - 100,000,000 → (50000000, 100000000)
0 - 20000 → (0, 20000)
nan → (nan, nan)


In [39]:
df[["owners_min", "owners_max"]] = df["estimated_owners"].apply(
    lambda s: pd.Series(parse_owners(s))
)

In [40]:
df["owners_mid"] = (
    df["owners_min"] + df["owners_max"]
) / 2

In [41]:
print(
    df[
        [
            "name",
            "estimated_owners",
            "owners_min",
            "owners_max",
            "owners_mid"
        ]
    ].head(10)
)

                              name       estimated_owners  owners_min  \
0                 Counter-Strike 2  100000000 - 200000000   100000000   
1              PUBG: BATTLEGROUNDS   50000000 - 100000000    50000000   
2                           Dota 2  200000000 - 500000000   200000000   
3        Grand Theft Auto V Legacy   50000000 - 100000000    50000000   
4  Tom Clancy's Rainbow Six® Siege              0 - 20000           0   
5  Tom Clancy's Rainbow Six® Siege              0 - 20000           0   
6  Tom Clancy's Rainbow Six® Siege              0 - 20000           0   
7  Tom Clancy's Rainbow Six® Siege              0 - 20000           0   
8  Tom Clancy's Rainbow Six® Siege    20000000 - 50000000    20000000   
9                  Team Fortress 2    20000000 - 50000000    20000000   

   owners_max   owners_mid  
0   200000000  150000000.0  
1   100000000   75000000.0  
2   500000000  350000000.0  
3   100000000   75000000.0  
4       20000      10000.0  
5       20000      100

In [42]:
for platform in ["windows", "mac", "linux"]:
    df[platform] = df[platform].astype(bool)

In [43]:
print(df[["windows", "mac", "linux"]].dtypes)

windows    bool
mac        bool
linux      bool
dtype: object


In [44]:
print("release_date:", df["release_date"].dtype)
print("release_year:", df["release_year"].dtype)

print("price:", df["price"].dtype)
print("is_free:", df["is_free"].dtype)
print("price_inr:", df["price_inr"].dtype)

print("owners_min:", df["owners_min"].dtype)
print("owners_max:", df["owners_max"].dtype)
print("owners_mid:", df["owners_mid"].dtype)

print("\nPlatforms:")
print(df[["windows", "mac", "linux"]].dtypes)

release_date: datetime64[ns]
release_year: int32
price: float64
is_free: bool
price_inr: float64
owners_min: int64
owners_max: int64
owners_mid: float64

Platforms:
windows    bool
mac        bool
linux      bool
dtype: object


In [45]:
def clean_text(t):
    if pd.isna(t):
        return ""

    t = html.unescape(str(t))
    t = re.sub(r"<[^>]+>", " ", t)
    t = re.sub(r"http\S+", " ", t)

    return re.sub(r"\s+", " ", t).strip()

In [46]:
test_texts = [
    "<h1>Amazing &amp; Fun Game</h1>",
    "Visit https://example.com for more information.",
    "Game    with\n\nmultiple     spaces.",
    np.nan,
    "<p>Explore <b>this</b> amazing world.</p>"
]

for text in test_texts:
    print("BEFORE:", repr(text))
    print("AFTER: ", repr(clean_text(text)))
    print("-" * 60)

BEFORE: '<h1>Amazing &amp; Fun Game</h1>'
AFTER:  'Amazing & Fun Game'
------------------------------------------------------------
BEFORE: 'Visit https://example.com for more information.'
AFTER:  'Visit for more information.'
------------------------------------------------------------
BEFORE: 'Game    with\n\nmultiple     spaces.'
AFTER:  'Game with multiple spaces.'
------------------------------------------------------------
BEFORE: nan
AFTER:  ''
------------------------------------------------------------
BEFORE: '<p>Explore <b>this</b> amazing world.</p>'
AFTER:  'Explore this amazing world.'
------------------------------------------------------------


In [47]:
text_columns = [
    "short_description",
    "about_the_game",
    "detailed_description"
]

for col in text_columns:
    df[col] = df[col].apply(clean_text)

In [48]:
for col in text_columns:
    print("=" * 60)
    print(col)
    print("Python type:", type(df[col].iloc[0]))
    print("Empty values:", (df[col] == "").sum())

short_description
Python type: <class 'str'>
Empty values: 85
about_the_game
Python type: <class 'str'>
Empty values: 185
detailed_description
Python type: <class 'str'>
Empty values: 162


In [49]:
mask = df["short_description"] == ""

df.loc[mask, "short_description"] = (
    df.loc[mask, "about_the_game"].str[:400]
)

In [50]:
print("Short descriptions using fallback:",
      mask.sum())

Short descriptions using fallback: 85


In [51]:
print(
    "Still empty after fallback:",
    (df["short_description"] == "").sum()
)

Still empty after fallback: 83


In [52]:
sample = df[
    ["name", "short_description", "about_the_game"]
].sample(5, random_state=42)

sample

,name,short_description,about_the_game
90073,Kana Seito Defense,Quickly learn and improve your Japanese langua...,Overview Kana Seito Defense is an educational ...
78473,Zombow,On an old farm there is something strange. The...,On an old farm there is something strange. The...
46280,Temporal Purge: Z,Become an interdimensional zombie slayer in Te...,"Temporal Purge: Z is a wave-based, single-play..."
42099,Haunted Hotel: The Thirteenth Collector's Edition,No good deed goes unpunished…,Elephant Games cordially invites you to visit ...
29549,Traumada,Welcome to the horrific altered memories of my...,"As we revisit our memories, they change. How d..."


In [53]:
rating_columns = [
    "positive",
    "negative",
    "num_reviews_total",
    "pct_pos_total"
]

for col in rating_columns:
    df[col] = pd.to_numeric(
        df[col],
        errors="coerce"
    )

In [54]:
print(
    "Missing pct_pos_total before:",
    df["pct_pos_total"].isna().sum()
)

Missing pct_pos_total before: 0


In [55]:
review_total = df["positive"] + df["negative"]

calculated_pct = np.where(
    review_total > 0,
    (df["positive"] / review_total) * 100,
    np.nan
)

df["pct_pos_total"] = df["pct_pos_total"].fillna(
    pd.Series(calculated_pct, index=df.index)
)

In [56]:
print(
    "Missing pct_pos_total after:",
    df["pct_pos_total"].isna().sum()
)

Missing pct_pos_total after: 0


In [57]:
print(
    "Metacritic score = 0:",
    (df["metacritic_score"] == 0).sum()
)

print(
    "User score = 0:",
    (df["user_score"] == 0).sum()
)

Metacritic score = 0: 85974
User score = 0: 89511


In [58]:
df["metacritic_score"] = (
    df["metacritic_score"]
    .replace(0, np.nan)
)

df["user_score"] = (
    df["user_score"]
    .replace(0, np.nan)
)

In [59]:
rating_check = df[
    [
        "positive",
        "negative",
        "num_reviews_total",
        "pct_pos_total",
        "metacritic_score",
        "user_score"
    ]
]

rating_check.info()

<class 'pandas.core.frame.DataFrame'>
Index: 89550 entries, 0 to 94947
Data columns (total 6 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   positive           89550 non-null  int64  
 1   negative           89550 non-null  int64  
 2   num_reviews_total  89550 non-null  int64  
 3   pct_pos_total      89550 non-null  int64  
 4   metacritic_score   3576 non-null   float64
 5   user_score         39 non-null     float64
dtypes: float64(2), int64(4)
memory usage: 4.8 MB


In [60]:
rating_check.describe()

,positive,negative,num_reviews_total,pct_pos_total,metacritic_score,user_score
count,8.955000e+04,8.955000e+04,8.955000e+04,89550.000000,3576.000000,39.000000
mean,1.281577e+03,2.140757e+02,1.388019e+03,45.386365,73.387304,75.410256
std,3.177024e+04,6.132221e+03,3.630640e+04,40.680714,10.207259,17.843940
min,0.000000e+00,0.000000e+00,-1.000000e+00,-1.000000,20.000000,37.000000
25%,1.000000e+00,0.000000e+00,-1.000000e+00,-1.000000,68.000000,62.000000
50%,1.000000e+01,2.000000e+00,1.500000e+01,60.000000,75.000000,77.000000
75%,5.800000e+01,1.700000e+01,8.100000e+01,84.000000,81.000000,90.000000
max,7.480813e+06,1.135108e+06,8.632939e+06,100.000000,97.000000,100.000000


In [61]:
from pathlib import Path

processed_dir = Path("../data/processed")
processed_dir.mkdir(parents=True, exist_ok=True)

In [85]:
output_path = processed_dir / "games_clean.parquet"

df.to_parquet(
    output_path,
    index=False
)

print("Clean dataset saved successfully!")
print("Path:", output_path)
print("Shape:", df.shape)

Clean dataset saved successfully!
Path: ..\data\processed\games_clean.parquet
Shape: (89550, 53)


In [63]:
df_check = pd.read_parquet(output_path)

print("Reloaded successfully!")
print("Shape:", df_check.shape)

Reloaded successfully!
Shape: (89550, 53)


In [64]:
print(
    "Same shape:",
    df.shape == df_check.shape
)

Same shape: True


In [65]:
print(
    "Same columns:",
    list(df.columns) == list(df_check.columns)
)

Same columns: True


In [66]:
list_columns = [
    "genres",
    "categories",
    "developers",
    "publishers",
    "supported_languages",
    "tags"
]

for col in list_columns:
    string_count = df[col].apply(
        lambda x: isinstance(x, str)
    ).sum()

    print(f"{col}: {string_count} string values")

genres: 0 string values
categories: 0 string values
developers: 0 string values
publishers: 0 string values
supported_languages: 0 string values
tags: 0 string values


In [67]:
print("release_date dtype:", df["release_date"].dtype)

release_date dtype: datetime64[ns]


In [68]:
print(
    "Is datetime:",
    pd.api.types.is_datetime64_any_dtype(df["release_date"])
)

Is datetime: True


In [69]:
final_missing = pd.DataFrame({
    "missing_count": df.isna().sum(),
    "missing_percentage": (
        df.isna().mean() * 100
    ).round(2)
})

final_missing = final_missing[
    final_missing["missing_count"] > 0
].sort_values(
    "missing_count",
    ascending=False
)

final_missing

,missing_count,missing_percentage
user_score,89511,99.96
score_rank,89511,99.96
metacritic_score,85974,96.01
metacritic_url,85974,96.01
reviews,79128,88.36
notes,72916,81.42
website,48438,54.09
support_url,45447,50.75
support_email,10816,12.08


In [86]:
cleaning_log_df = pd.DataFrame(cleaning_log)

cleaning_log_df

,step,rows_before,rows_after,rows_removed
0,Remove rows with missing name/appid,94948,94946,2
1,Remove duplicate appid values,94946,94946,0
2,Remove non-game entries,94946,89550,5396
3,Remove entries with no description and no genres,89550,89550,0


In [87]:
data_dictionary = pd.DataFrame([
    ["appid", "int", "Unique Steam application identifier"],
    ["name", "string", "Game name"],
    ["release_date", "datetime", "Game release date"],
    ["release_year", "numeric", "Year extracted from release_date"],
    ["price", "numeric", "Original dataset price"],
    ["price_inr", "numeric", "Price converted using project USD_TO_INR"],
    ["is_free", "boolean", "True when price is zero"],
    ["genres", "list", "Game genres"],
    ["categories", "list", "Steam categories/features"],
    ["tags", "list", "Top 15 tags by vote count"],
    ["developers", "list", "Game developers"],
    ["publishers", "list", "Game publishers"],
    ["supported_languages", "list", "Supported languages"],
    ["estimated_owners", "string", "Original estimated owner range"],
    ["owners_min", "numeric", "Lower bound of estimated owners"],
    ["owners_max", "numeric", "Upper bound of estimated owners"],
    ["owners_mid", "numeric", "Midpoint of estimated owner range"],
    ["positive", "numeric", "Positive review count"],
    ["negative", "numeric", "Negative review count"],
    ["num_reviews_total", "numeric", "Total review count"],
    ["pct_pos_total", "numeric", "Percentage of positive reviews"],
    ["metacritic_score", "numeric", "Metacritic score; unavailable placeholders treated as missing"],
    ["user_score", "numeric", "User score; unavailable placeholders treated as missing"],
    ["short_description", "string", "Cleaned primary game description"],
    ["about_the_game", "string", "Cleaned detailed game information"],
    ["detailed_description", "string", "Cleaned detailed description"],
    ["windows", "boolean", "Windows platform support"],
    ["mac", "boolean", "macOS platform support"],
    ["linux", "boolean", "Linux platform support"],
], columns=["column", "type", "description"])

data_dictionary

,column,type,description
0,appid,int,Unique Steam application identifier
1,name,string,Game name
2,release_date,datetime,Game release date
3,release_year,numeric,Year extracted from release_date
4,price,numeric,Original dataset price
5,price_inr,numeric,Price converted using project USD_TO_INR
6,is_free,boolean,True when price is zero
7,genres,list,Game genres
8,categories,list,Steam categories/features
9,tags,list,Top 15 tags by vote count


In [88]:
cleaning_log_df.to_csv(
    processed_dir / "cleaning_log.csv",
    index=False
)

data_dictionary.to_csv(
    processed_dir / "data_dictionary.csv",
    index=False
)

print("Cleaning log and data dictionary saved.")

Cleaning log and data dictionary saved.


In [73]:
print("=" * 70)
print("GAMEGUIDE — STEP 2 FINAL VALIDATION")
print("=" * 70)

print("\nDataset:")
print("Rows:", len(df))
print("Columns:", len(df.columns))

print("\nList columns:")
for col in list_columns:
    is_string = df[col].apply(lambda x: isinstance(x, str)).sum()
    print(f"{col}: {is_string} string values")

print("\nDate:")
print("release_date dtype:", df["release_date"].dtype)
print(
    "Is datetime:",
    pd.api.types.is_datetime64_any_dtype(df["release_date"])
)

print("\nMissing values:")
print(
    "Columns with missing values:",
    (df.isna().sum() > 0).sum()
)

print("\nCleaning log:")
display(cleaning_log_df)

print("\nSaved dataset:")
print(output_path)

print("\nFinal shape matches saved dataset:",
      df.shape == df_check.shape)

print("=" * 70)

GAMEGUIDE — STEP 2 FINAL VALIDATION

Dataset:
Rows: 89550
Columns: 53

List columns:
genres: 0 string values
categories: 0 string values
developers: 0 string values
publishers: 0 string values
supported_languages: 0 string values
tags: 0 string values

Date:
release_date dtype: datetime64[ns]
Is datetime: True

Missing values:
Columns with missing values: 9

Cleaning log:


,step,rows_before,rows_after,rows_removed
0,Remove rows with missing name/appid,94948,94946,2
1,Remove duplicate appid values,94946,94946,0
2,Remove non-game entries,94946,89550,5396
3,Remove entries with no description and no genres,89550,89550,0



Saved dataset:
..\data\processed\games_clean.parquet

Final shape matches saved dataset: True


In [81]:
print(
    "num_reviews_total = -1:",
    (df["num_reviews_total"] == -1).sum()
)

print(
    "pct_pos_total = -1:",
    (df["pct_pos_total"] == -1).sum()
)

num_reviews_total = -1: 36352
pct_pos_total = -1: 36352


In [82]:
print(
    "positive = -1:",
    (df["positive"] == -1).sum()
)

print(
    "negative = -1:",
    (df["negative"] == -1).sum()
)

positive = -1: 0
negative = -1: 0


In [83]:
# Treat -1 as a missing-value placeholder for review statistics
df["num_reviews_total"] = df["num_reviews_total"].replace(-1, np.nan)
df["pct_pos_total"] = df["pct_pos_total"].replace(-1, np.nan)

In [84]:
print("num_reviews_total missing:", df["num_reviews_total"].isna().sum())
print("pct_pos_total missing:", df["pct_pos_total"].isna().sum())

num_reviews_total missing: 36352
pct_pos_total missing: 36352


# Step 2 — Data Cleaning & Preparation Complete

The raw Steam games dataset has been cleaned and prepared for the next stage of the GameGuide project.

### Completed Tasks

- Removed rows with missing game names and duplicate `appid` values.
- Removed identified non-game entries such as demos, playtests, soundtracks, artbooks, wallpapers, and dedicated servers.
- Converted list-like columns into proper Python lists.
- Parsed the `tags` column and retained the top 15 tags by vote count.
- Converted `release_date` to datetime and created `release_year`.
- Normalized game prices and created:
  - `is_free`
  - `price_inr`
- Parsed `estimated_owners` into:
  - `owners_min`
  - `owners_max`
  - `owners_mid`
- Standardized platform columns (`windows`, `mac`, `linux`) as boolean values.
- Cleaned HTML, URLs, and whitespace from game descriptions.
- Used `about_the_game` as a fallback when `short_description` was unavailable.
- Cleaned rating fields and converted placeholder `-1` values to missing values where appropriate.
- Treated zero placeholder values in `metacritic_score` and `user_score` as missing.
- Saved the cleaned dataset as:

`data/processed/games_clean.parquet`

### Final Dataset

- **Rows:** 89,550
- **Columns:** 53
- **Output format:** Parquet

The cleaned dataset is now ready for efficient lookup, indexing, search, recommendation, and chatbot development.
